# Telugu Text Classifier — Phase 2 training (Colab GPU)

Fine-tunes IndicBERTv2 / MuRIL / XLM-R on the frozen Phase 1 splits.

**Before running:** `Runtime > Change runtime type > T4 GPU`, and set `REPO_URL` below (the repo must be pushed to GitHub).
Total: 9 seed runs + 1 LR comparison ≈ 2–2.5 h on a T4. Checkpoints land on Google Drive so they survive disconnects.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/Telugu-Text-Classifier"  # <-- EDIT
DRIVE_DIR = "/content/drive/MyDrive/telugu-classifier"               # checkpoints + results land here

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os
if not os.path.exists("/content/Telugu-Text-Classifier"):
    !git clone {REPO_URL} /content/Telugu-Text-Classifier
%cd /content/Telugu-Text-Classifier
# Colab already ships CUDA torch; requirements.txt intentionally leaves torch unpinned
!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_DIR}/outputs", exist_ok=True)
os.makedirs("results/runs", exist_ok=True)
print("checkpoints ->", f"{DRIVE_DIR}/outputs")

In [ ]:
# Checkpoint root lives on Drive: survives runtime disconnects
!rm -rf outputs && ln -s "{DRIVE_DIR}/outputs" outputs
# Reproduces data/processed/{train,validation,test}.csv exactly (seeded Phase 1 pipeline)
!python scripts/prepare_data.py

In [ ]:
# 3 models x 3 seeds, sequential. Runs detached so the cell returns immediately.
!nohup python scripts/run_experiments.py > train.log 2>&1 &
print("launched; poll with the next cell")

In [ ]:
!tail -5 train.log
!ls results/runs/ 2>/dev/null

In [ ]:
# After all 9 runs: LR comparison on the best model, selected on VALIDATION only.
import json, glob, pathlib, statistics
by_model = {}
for p in glob.glob("results/runs/*.json"):
    s = json.loads(pathlib.Path(p).read_text())
    by_model.setdefault(s["model"], []).append(s["best_val_macro_f1"])
means = {m: statistics.mean(v) for m, v in by_model.items()}
best_model = max(means, key=means.get)
print({m: round(v, 4) for m, v in means.items()}, "-> best:", best_model)
!nohup python scripts/run_experiments.py --models {best_model} --seeds 42 --lr 5e-5 > train_lr.log 2>&1 &
print("launched lr=5e-5 run for", best_model)

In [ ]:
# Test-set evaluation, once per final model:
# best seed (by val macro-F1) at the best LR (by val macro-F1).
import json, glob, pathlib, subprocess
runs = {}
for p in glob.glob("results/runs/*.json"):
    s = json.loads(pathlib.Path(p).read_text())
    runs.setdefault(s["model"], []).append(s)
for model, rs in runs.items():
    best_lr = max(r["best_val_macro_f1"] for r in rs if r["lr"] == 5e-5, default=None)
    final = max((r for r in rs if (best_lr is None or r["lr"] == 5e-5)), key=lambda r: r["best_val_macro_f1"])
    name = model.split("/")[-1]
    print(">>>", model, "lr", final["lr"], "seed", final["seed"])
    r = subprocess.run(["python", "scripts/eval_transformer.py",
                        "--model-dir", f"outputs/{name}_seed{final['seed']}",
                        "--name", name], capture_output=True, text=True)
    print(r.stdout[-300:] or r.stderr[-1500:])

In [ ]:
!python scripts/compare_models.py
!python scripts/export_errors_analysis.py
!cp results/transformer_*_test_metrics.json results/runs/*.json "{DRIVE_DIR}/" 
!cp results/comparison_*.png results/*confusion*.png reports/comparison.md "{DRIVE_DIR}/"
!ls "{DRIVE_DIR}/"